# ADN du Dataset : Single Model Theory
Ce notebook implémente la stratégie de modèle unique optimisée, utilisant la physique des données (contraintes de monotonie) et une optimisation de seuil pour maximiser le score F1.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

SEED = 42
print("Setup Complete.")

Setup Complete.


In [2]:
# 1. Chargement des données
df_train = pd.read_csv('conversion_data_train.csv')
df_test = pd.read_csv('conversion_data_test.csv')

print(f"Train set: {df_train.shape}")
print(f"Test set: {df_test.shape}")

Train set: (284580, 6)
Test set: (31620, 5)


In [3]:
# 2. Prétraitement Minimal (Label Encoding)
features = ['country', 'source', 'new_user', 'age', 'total_pages_visited']

for col in ['country', 'source']:
    le = LabelEncoder()
    full = pd.concat([df_train[col], df_test[col]])
    le.fit(full)
    df_train[col] = le.transform(df_train[col])
    df_test[col] = le.transform(df_test[col])

X = df_train[features]
y = df_train['converted']
X_test = df_test[features]

print("Preprocessing terminé.")

Preprocessing terminé.


In [4]:
# 3. Configuration du Modèle (Paramètres optimisés)
params = {
    'learning_rate': 0.01,
    'max_depth': 6,
    'n_estimators': 500,
    'subsample': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'n_jobs': -1,
    'random_state': SEED,
    'monotone_constraints': '(0, 0, 0, -1, 1)'  # Âge diminue, Pages augmente la conversion
}

print("Entraînement du modèle final...")
model = xgb.XGBClassifier(**params)
model.fit(X, y)
print("Modèle entraîné.")

Entraînement du modèle final...
Modèle entraîné.


In [ ]:
# 4. Optimisation du Seuil sur le Train Set
probs_train = model.predict_proba(X)[:, 1]
best_f1 = 0
best_th = 0.5

print("Optimisation du seuil en cours...")
for th in np.arange(0.3, 0.6, 0.005):
    f1 = f1_score(y, (probs_train >= th).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_th = th
        
print(f"Seuil optimal trouvé : {best_th:.3f}")
print(f"Score F1 sur train : {best_f1:.5f}")

Optimisation du seuil en cours...
Seuil optimal trouvé : 0.385
Score F1 sur train : 0.77109


In [ ]:
# 5. Prédictions Finales et Export
probs_test = model.predict_proba(X_test)[:, 1]
preds_test = (probs_test >= best_th).astype(int)

submission = pd.DataFrame({'converted': preds_test})
submission.to_csv('submission_adn_tosophilippe.csv', index=False)

print("Fichier généré : submission_adn_tosophilippe.csv")
print(f"Total des conversions prédites : {submission['converted'].sum()}")

Fichier généré : submission_adn_tosophilippe.csv
Total des conversions prédites : 936
